# J1-02 · Les fondamentaux des agents : outils, boucle ReAct, `create_agent`

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 1 · créneau 10h45–12h00 · ⏱️ ~1h15 (dont 45 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Écrire un outil (`@tool`) que le LLM comprend · 2. Voir ce que le LLM reçoit réellement (schéma JSON) · 3. Coder la boucle ReAct **à la main** · 4. Obtenir la même chose avec `create_agent` · 5. Lire dans Langfuse ce que l'agent a fait, tour par tour |
| 🧱 **Fil rouge** | HelpDesk-Ops passe du **classifieur** (J1-01) à un **agent** qui interroge l'inventaire de NordTech SA |
| ✅ **Prérequis** | J1-01 terminé, stack démarrée (`uv run smoke_test.py` au vert) |
| 📦 **Versions** | `langchain 1.4` · `langgraph 1.2` · `langfuse 4.15` (SDK v4), figées par `uv.lock` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Qu'est-ce qu'un agent ?

### [Théorie] Un LLM qui choisit ses outils et décide quand il a fini

*Analogie : la recette et le chef.* Une chaîne LCEL suit une **recette** : les étapes sont écrites à l'avance. Un agent se comporte comme un **chef** : il goûte, décide d'ajouter du sel, goûte encore, puis décide que c'est prêt.

Un agent, c'est quatre éléments :

| Élément | Rôle | Dans HelpDesk-Ops |
|---|---|---|
| **LLM** | décide de la prochaine action | `granite4.1:8b` |
| **Outils** | fonctions Python décrites au LLM | `statut_serveur`, `tickets_ouverts` |
| **Boucle** | décider → agir → observer, jusqu'à la réponse | ReAct |
| **Condition d'arrêt** | le LLM ne demande plus d'outil… ou une limite l'arrête | `max` de tours, `recursion_limit` |

> 💡 **Le « Tips » du Formateur :** la condition d'arrêt est la source n°1 des incidents d'agents en production (boucles qui consomment des centaines d'appels). On la traite **dès maintenant**, pas « plus tard ».

## 3. Écrire un outil que le LLM comprend

### [Théorie] Le LLM ne voit jamais votre code

Il ne reçoit que trois choses : le **nom** de la fonction, sa **description** (la docstring) et le **schéma JSON** de ses arguments (déduit des *type hints*). Une docstring vague, c'est un outil mal utilisé.

> 🛡️ **OWASP ASI02 (Tool Misuse) :** chaque outil exposé est une surface d'attaque. Exposez le **minimum** : ici, des outils en **lecture seule**.

In [ ]:
from langchain_core.tools import tool

# Inventaire fictif de NordTech SA (en vrai : une CMDB, une API de supervision…)
INVENTAIRE = {
    "srv-mail-01": "en ligne",
    "srv-erp-02": "hors ligne",
    "srv-vpn-01": "en ligne",
    "srv-fichiers-03": "dégradé (disque à 92 %)",
}
TICKETS_OUVERTS = {"N1": 14, "N2-infra": 6, "N2-applicatif": 3}


@tool(parse_docstring=True)
def statut_serveur(nom: str) -> str:
    """Retourne le statut (en ligne, hors ligne, dégradé) d'un serveur de l'inventaire NordTech.

    Args:
        nom: nom d'hôte exact en minuscules, par exemple 'srv-mail-01'
    """
    # Normaliser l'entrée : le LLM peut écrire 'SRV-MAIL-01 ' ; une erreur explicite vaut mieux qu'un None
    return INVENTAIRE.get(nom.strip().lower(), f"inconnu : '{nom}' n'est pas dans l'inventaire")


@tool(parse_docstring=True)
def tickets_ouverts(file: str) -> str:
    """Retourne le nombre de tickets ouverts dans une file d'attente du help-desk.

    Args:
        file: nom exact de la file : 'N1', 'N2-infra' ou 'N2-applicatif'
    """
    if file not in TICKETS_OUVERTS:
        return f"file inconnue : '{file}'. Files valides : {', '.join(TICKETS_OUVERTS)}"
    return f"{TICKETS_OUVERTS[file]} tickets ouverts dans la file {file}"


# Un outil reste une fonction : on le teste sans LLM
print(statut_serveur.invoke({"nom": "SRV-ERP-02"}))
print(tickets_ouverts.invoke({"file": "N2-infra"}))

### [Code] Ce que le LLM reçoit réellement

`convert_to_openai_tool` produit le schéma JSON envoyé au modèle avec **chaque** requête. Comparons avec et sans `parse_docstring=True`.

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(statut_serveur), indent=2, ensure_ascii=False))


# Même fonction, décorée sans parse_docstring : la description de l'argument disparaît
@tool
def statut_serveur_sans_args(nom: str) -> str:
    """Retourne le statut d'un serveur de l'inventaire NordTech.

    Args:
        nom: nom d'hôte exact en minuscules, par exemple 'srv-mail-01'
    """
    return INVENTAIRE.get(nom, "inconnu")


print("\nSans parse_docstring :", convert_to_openai_tool(statut_serveur_sans_args)["function"]["parameters"]["properties"])

**👀 Ce que vous devez observer :** avec `parse_docstring=True`, l'argument `nom` porte sa description (`nom d'hôte exact…`). Sans, il n'a qu'un type `string` : le LLM doit deviner le format attendu.

## 4. `bind_tools` : le LLM *demande*, il n'exécute pas

### [Théorie] Un `AIMessage` avec des `tool_calls`

`llm.bind_tools([...])` attache les schémas à chaque requête. Le modèle peut alors répondre par un **`AIMessage` contenant des `tool_calls`** : « appelle `statut_serveur` avec `nom='srv-erp-02'` ». **C'est votre code qui exécute l'outil**, puis renvoie le résultat dans un `ToolMessage` lié par `tool_call_id`.

*Analogie : le chef dicte, le commis exécute.* Le chef (LLM) ne touche pas aux fourneaux ; il donne des ordres précis, le commis (votre code) les exécute et lui rapporte le résultat.

In [ ]:
TOOLS = {t.name: t for t in (statut_serveur, tickets_ouverts)}
llm_outils = llm.bind_tools(list(TOOLS.values()))

reponse = llm_outils.invoke(
    "Le serveur srv-erp-02 répond-il ?",
    config={"callbacks": [langfuse_handler], "run_name": "bind-tools-demo"},
)
print("content    :", repr(reponse.content))
print("tool_calls :", reponse.tool_calls)

**👀 Ce que vous devez observer :** `content` est vide (ou presque) et `tool_calls` contient `{'name': 'statut_serveur', 'args': {'nom': 'srv-erp-02'}, 'id': '…'}`. Le modèle n'a **pas** la réponse : il demande une action.

*Pas de `tool_calls` ?* Le modèle n'a pas la capacité « tools » ou la question est ambiguë. Vérifiez avec `docker compose exec ollama ollama show <modèle>` (section *Capabilities*).

## 5. La boucle ReAct, codée à la main

### [Théorie] Raisonner → Agir → Observer

ReAct (*Reason + Act*, Yao et al., 2022) : le modèle alterne raisonnement et actions jusqu'à pouvoir répondre. On la code **à la main** une fois, pour que le graphe de J1-03 paraisse évident par contraste.

> 💡 **Le « Tips » du Formateur :** **tout l'historique** repart au LLM à chaque tour. Coût et taille du contexte augmentent à chaque boucle : c'est pour ça que `num_ctx` compte.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

PROMPT_SYSTEME = (
    "Tu es l'agent support N1 de NordTech SA. Réponds en français, en une ou deux phrases. "
    "Utilise les outils pour toute information sur les serveurs ou les files de tickets ; n'invente rien."
)


@observe(name="react-manuel")   # une trace racine qui regroupe tous les tours
def agent_react_manuel(question: str, max_tours: int = 5) -> str:
    messages = [SystemMessage(PROMPT_SYSTEME), HumanMessage(question)]
    for tour in range(1, max_tours + 1):                 # garde-fou : jamais de boucle infinie
        reponse = llm_outils.invoke(messages, config={"callbacks": [langfuse_handler]})
        messages.append(reponse)
        if not reponse.tool_calls:                       # plus d'outil demandé -> réponse finale
            print(f"  ✔ réponse au tour {tour}")
            return reponse.content
        for appel in reponse.tool_calls:                 # exécuter chaque outil demandé
            print(f"  ↳ tour {tour} : {appel['name']}({appel['args']})")
            resultat = TOOLS[appel["name"]].invoke(appel["args"], config={"callbacks": [langfuse_handler]})
            messages.append(ToolMessage(content=resultat, tool_call_id=appel["id"]))
    return "Je n'ai pas pu conclure : je transmets votre demande à un technicien."   # repli explicite


with propagate_attributes(session_id="j1-02-react", tags=["jour-1", "react-manuel"]):
    print(agent_react_manuel("Le serveur srv-erp-02 est-il en ligne ? Et combien de tickets attendent en N2-infra ?"))
langfuse.flush()

**👀 Ce que vous devez observer :**
- dans la sortie : un ou deux tours d'outils, puis la réponse ;
- dans Langfuse (**Tracing → `react-manuel`**) : une generation par tour, les observations `statut_serveur` / `tickets_ouverts` avec leurs arguments, et le Gantt. Comptez les generations : c'est le **nombre de tours**.

**Question pour la salle :** que manque-t-il à ces 20 lignes pour aller en production ? *(reprise après un crash, pause pour validation humaine, parallélisme, traçabilité de chaque décision…)*

## 6. La même chose en un appel : `create_agent`

### [Théorie] La boucle, empaquetée

`create_agent` (LangChain 1.x) construit cette boucle pour vous. Sous le capot, c'est un **`StateGraph` LangGraph compilé** avec deux nœuds, `model` et `tools` : exactement ce qu'on dessinera en J1-03.

> ⚠️ **Tutoriels obsolètes :** `AgentExecutor`, `initialize_agent` (LangChain < 1.0) et `create_react_agent` de `langgraph.prebuilt` (brique de bas niveau) ne sont plus la voie recommandée. Retenez le **concept** (la boucle), pas le nom de la fonction : il a changé trois fois.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[statut_serveur, tickets_ouverts],
    system_prompt=PROMPT_SYSTEME,
)

with propagate_attributes(session_id="j1-02-react", tags=["jour-1", "create-agent"]):
    resultat = agent.invoke(
        {"messages": [{"role": "user", "content": "Le serveur srv-erp-02 est-il en ligne ? Et combien de tickets attendent en N2-infra ?"}]},
        config={"callbacks": [langfuse_handler], "run_name": "create-agent", "recursion_limit": 10},
    )

for message in resultat["messages"]:
    print(f"{type(message).__name__:<13} {message.content[:90]!r} {getattr(message, 'tool_calls', '') or ''}")
print("\nType de l'agent :", type(agent).__name__)
langfuse.flush()

**👀 Ce que vous devez observer :**
- la séquence `HumanMessage → AIMessage (tool_calls) → ToolMessage… → AIMessage (réponse)` : c'est la conversation de l'agent, message par message ;
- le type : `CompiledStateGraph`. Un agent **est** un graphe ;
- dans Langfuse, la trace `create-agent` a la même forme que `react-manuel`, avec les nœuds `model` et `tools` en plus.

## 7. Les limites d'un petit modèle, mesurées

### [Théorie] Le tool-calling n'est pas binaire

Un petit modèle « supporte les outils »… plus ou moins bien : outil oublié, argument mal formé, réponse inventée. On ne le juge pas à l'impression, on le **regarde dans les traces** (et on le mesurera en J4).

> 💡 **Le « Tips » du Formateur :** sur un poste sans GPU, cette cellule peut prendre une minute. Lancez-la, puis commentez la trace pendant qu'elle tourne chez les autres.

In [ ]:
petit_llm = creer_llm("small")   # MODEL_SMALL
petit_agent = create_agent(model=petit_llm, tools=[statut_serveur, tickets_ouverts], system_prompt=PROMPT_SYSTEME)

question_piege = "Donne-moi le statut de SRV-FICHIERS-03 et le nombre de tickets de la file infra de niveau 2."
for nom, a in (("MODEL_AGENT", agent), ("MODEL_SMALL", petit_agent)):
    with propagate_attributes(session_id="j1-02-comparaison", tags=["jour-1", "comparaison"], metadata={"role": nom}):
        r = a.invoke(
            {"messages": [{"role": "user", "content": question_piege}]},
            config={"callbacks": [langfuse_handler], "run_name": f"comparaison-{nom}", "recursion_limit": 10},
        )
    appels = [c for m in r["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    print(f"{nom:<12} {len(appels)} appel(s) d'outil : {[(c['name'], c['args']) for c in appels]}")
    print(f"{'':<12} → {r['messages'][-1].content[:150]}\n")
langfuse.flush()

**👀 Ce que vous devez observer :** la question est volontairement piégeuse (majuscules, « file infra de niveau 2 » au lieu de `N2-infra`). Comparez dans **Sessions → `j1-02-comparaison`** : nombre d'appels, arguments envoyés, messages d'erreur renvoyés par l'outil, réponse finale. Un outil qui renvoie une **erreur explicite** (« file inconnue… files valides : … ») permet souvent au modèle de se corriger au tour suivant.

*Les deux modèles réussissent ?* Tant mieux : relancez avec une question plus ambiguë. La leçon reste : **on vérifie dans la trace**, pas à l'œil.

## 8. Les 6 pièges classiques des agents locaux

| Piège | Symptôme | Parade |
|---|---|---|
| 🐣 Petit modèle | `tool_calls` absents ou mal formés | mesurer (J4) avant de conclure ; erreurs d'outil explicites |
| ♾️ Boucle infinie | le modèle redemande sans cesse un outil | `max_tours` / `recursion_limit` explicite |
| 📝 Docstring vague | mauvais outil choisi | description précise + exemples d'arguments |
| 🧰 Trop d'outils | au-delà d'une dizaine, le choix se dégrade | regrouper, ou déléguer à des sous-agents (J3) |
| ☣️ Outil dangereux | `eval()`, shell, SQL libre (OWASP ASI05) | liste blanche, validation (J5) |
| 📦 Contexte saturé | instructions système « oubliées » | `num_ctx` suffisant, résumé de l'historique (J5) |

## 9. 🧪 Mini-défi (10 min)

1. Ajoutez un outil `redemarrer_serveur(nom: str) -> str` qui **ne redémarre rien** : il renvoie « redémarrage demandé pour <nom>, en attente de validation ».
2. Donnez-le à un nouvel agent et demandez : « srv-erp-02 est hors ligne, fais le nécessaire ».
3. Dans Langfuse, vérifiez si l'agent a appelé l'outil **sans vous demander confirmation**.
4. Discussion : quels outils devraient exiger une **validation humaine** ? (→ HITL, J2)

In [ ]:
# À vous de jouer 👇

## 10. Avant de quitter : `flush()`

In [ ]:
langfuse.flush()
print("✅ Observations envoyées : http://localhost:3000")

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Écrire un outil compréhensible par le LLM | `@tool(parse_docstring=True)` + docstring précise |
| Voir ce que le LLM reçoit | `convert_to_openai_tool(outil)` |
| Faire demander une action au LLM | `llm.bind_tools([...])` → `AIMessage.tool_calls` |
| Coder la boucle ReAct | `ToolMessage(content=…, tool_call_id=…)`, `max_tours` |
| Créer un agent en un appel | `create_agent(model, tools, system_prompt=…)` |
| Tracer l'ensemble | `@observe`, `propagate_attributes`, `CallbackHandler` |

**Prochain notebook, J1-03 :** on dessine cette boucle nous-mêmes avec `StateGraph`.

### ✅ Quiz express

1. Qu'est-ce que le LLM « voit » d'un outil ?
2. Qui exécute l'outil : le LLM ou votre code ?
3. Quelle est la condition d'arrêt naturelle de la boucle ReAct, et pourquoi ne suffit-elle pas ?
4. Que renvoie `create_agent`, techniquement ?

<details>
<summary>Réponses</summary>

1. Son nom, sa description (docstring) et le schéma JSON de ses arguments. Jamais son code.
2. Votre code. Le LLM produit seulement une demande (`tool_calls`).
3. L'`AIMessage` ne contient plus de `tool_calls`. Un modèle peut redemander un outil indéfiniment : il faut une limite explicite (`max_tours`, `recursion_limit`).
4. Un `CompiledStateGraph` LangGraph (nœuds `model` et `tools`).
</details>